# НИРМ 4 — предлагаемый метод: пол на уровне отведений; сравнение с известными подходами (Google Colab)

Продолжение `nirm4_sexaware_colab.ipynb`. Там проверены возраст и пол на входе головы и веса по полу —
разрыв качества между мужчинами и женщинами не сократился. Неделя 9 показала, что у женщин модель иначе
использует **отведения** (меньше V4–V5) и комплекс QRS, поэтому пол подаётся туда, где возникает различие.

Новые варианты CNN-LSTM (архитектура и гиперпараметры недели 8, по 5 seed):
- `leadgate` — **предлагаемый метод**: пол и возраст задают коэффициент усиления каждого из 12 отведений до
  свёрток (g = 2·sigmoid(MLP(возраст, пол)), начально g = 1). Коэффициенты интерпретируемы: видно, какие
  отведения модель усиливает у женщин и у мужчин;
- `film` — пол и возраст модулируют каналы свёрточного стебля (FiLM, Perez et al., 2018);
- `sexnorm` — гипотеза амплитуды: нормировка отведений статистикой своего пола (обучающие фолды);
- `adv` — известный конкурирующий подход: adversarial debiasing (обращение градиента, Ganin & Lempitsky,
  2015) — из скрытого представления удаляется информация о поле.

Варианты `base`, `demo`, `groupw`, `demo_groupw` уже обучены и пропускаются, если папка `results_nirm4/`
на месте. **Перед запуском** ничего догружать не нужно: используются те же `X100_bp05-40_crop.npy` и
`colab_meta_v2.npz`. T4 GPU → `Run all`; 20 новых запусков, около 30–40 минут.

In [ ]:
import json
import os
import sys
import time

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

IN_COLAB = 'google.colab' in sys.modules
SMOKE = os.environ.get('SMOKE') == '1'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CANDIDATES = ['/content/drive/MyDrive/thesis_ptbxl/', '/content/drive/MyDrive/Colab Notebooks/thesis_ptbxl/']
    ROOT = next((d for d in CANDIDATES if os.path.exists(d + 'colab_meta_v2.npz')), None)
    assert ROOT, f'не найден colab_meta_v2.npz ни в одной из папок: {CANDIDATES}'
else:
    ROOT = 'cache/'
OUT = os.environ.get('COLAB_OUT', ROOT)
RESULTS, CKPT = OUT + 'results_nirm4/', OUT + 'ckpt_nirm4/'
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(CKPT, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
ENV = {'torch': torch.__version__, 'cuda': torch.version.cuda, 'device': DEVICE,
       'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None}
print(ENV)

In [ ]:
meta = np.load(ROOT + 'colab_meta_v2.npz')
X_all = np.load(ROOT + 'X100_bp05-40_crop.npy', mmap_mode='r')
Y_all, split, has_label, sex = meta['Y'].astype(np.float32), meta['split'], meta['has_label'], meta['sex']
LEAD_MEAN, LEAD_STD = meta['lead_mean'].astype(np.float32), meta['lead_std'].astype(np.float32)
FS = int(meta['fs'])
idx = {s: np.where((split == s) & has_label)[0] for s in ('train', 'val', 'test')}
if SMOKE:
    idx = {s: v[:{'train': 600, 'val': 300, 'test': 300}[s]] for s, v in idx.items()}

age = meta['age'].astype(np.float32)
AGE_MEAN, AGE_STD = age[idx['train']].mean(), age[idx['train']].std()
DEMO_ALL = np.stack([(age - AGE_MEAN) / AGE_STD, (sex == 'female').astype(np.float32)], axis=1)


def group_weights(rows):
    y, female = Y_all[rows], sex[rows] == 'female'
    w = np.ones_like(y)
    for k in range(y.shape[1]):
        for label in (0, 1):
            n_label = (y[:, k] == label).sum()
            for grp in (female, ~female):
                m = grp & (y[:, k] == label)
                w[m, k] = n_label / (2 * m.sum())
    return w


def to_device(rows, mean=None, std=None):
    """Нормировка по отведениям; mean/std формы (n, 1, leads) — для нормировки статистикой своего пола."""
    mean = LEAD_MEAN if mean is None else mean
    std = LEAD_STD if std is None else std
    x = (np.asarray(X_all[rows]) - mean) / std
    return torch.from_numpy(x.astype(np.float32)).to(DEVICE)


def build_data(norm='global'):
    if norm == 'global':
        stats = {s: (None, None) for s in idx}
    else:
        tr = idx['train']
        by_sex = {g: (np.asarray(X_all[tr[sex[tr] == g]]).mean(axis=(0, 1)),
                      np.asarray(X_all[tr[sex[tr] == g]]).std(axis=(0, 1))) for g in ('female', 'male')}
        print('СКО отведений, женщины / мужчины:', np.round(by_sex['female'][1] / by_sex['male'][1], 3))
        stats = {}
        for s, r in idx.items():
            f = (sex[r] == 'female')[:, None, None]
            stats[s] = (np.where(f, by_sex['female'][0], by_sex['male'][0]).astype(np.float32),
                        np.where(f, by_sex['female'][1], by_sex['male'][1]).astype(np.float32))
    return {s: (to_device(r, *stats[s]), torch.from_numpy(Y_all[r]).to(DEVICE),
                torch.from_numpy(DEMO_ALL[r]).to(DEVICE)) for s, r in idx.items()}


DATA_CACHE = {'global': build_data('global')}
W_TRAIN = torch.from_numpy(group_weights(idx['train'])).to(DEVICE)


def get_data(norm):
    if norm not in DATA_CACHE:
        DATA_CACHE[norm] = build_data(norm)
    return DATA_CACHE[norm]


print({s: tuple(x.shape) for s, (x, _, _) in DATA_CACHE['global'].items()})

## Модель

`CNNLSTM` без дополнительных модулей — ровно модель недели 8 (тот же порядок создания слоёв, те же веса при
том же seed). Новые модули создаются после всех прежних и инициализируются так, что в начале обучения сеть
совпадает с базовой: усиление отведений g = 1, модуляция FiLM нулевая.

In [ ]:
def conv_block(c_in, c_out, k=7, pool=True):
    layers = [nn.Conv1d(c_in, c_out, k, padding='same', bias=False), nn.BatchNorm1d(c_out), nn.ReLU(),
              nn.Conv1d(c_out, c_out, k, padding='same', bias=False), nn.BatchNorm1d(c_out), nn.ReLU()]
    return nn.Sequential(*layers, nn.MaxPool1d(2)) if pool else nn.Sequential(*layers)


class AttentionPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))
        self.weights = None

    def forward(self, h):
        w = torch.softmax(self.score(h).squeeze(-1), dim=1)
        self.weights = w.detach()
        return (w.unsqueeze(-1) * h).sum(dim=1)


def conv_stem(width, n_leads=12):
    return nn.Sequential(conv_block(n_leads, width // 2), conv_block(width // 2, width))


def zero_last(seq):
    nn.init.zeros_(seq[-1].weight)
    nn.init.zeros_(seq[-1].bias)
    return seq


class LeadGate(nn.Module):
    """Предлагаемый модуль: коэффициент усиления каждого отведения как функция пола и возраста."""

    def __init__(self, n_leads=12, hidden=16):
        super().__init__()
        self.net = zero_last(nn.Sequential(nn.Linear(2, hidden), nn.ReLU(), nn.Linear(hidden, n_leads)))
        self.gains = None

    def forward(self, x, demo):  # x: (batch, leads, time)
        g = 2 * torch.sigmoid(self.net(demo))
        self.gains = g.detach()
        return x * g.unsqueeze(-1)


class FiLM(nn.Module):
    def __init__(self, channels, hidden=16):
        super().__init__()
        self.net = zero_last(nn.Sequential(nn.Linear(2, hidden), nn.ReLU(), nn.Linear(hidden, 2 * channels)))

    def forward(self, h, demo):  # h: (batch, channels, time)
        gamma, beta = self.net(demo).chunk(2, dim=1)
        return h * (1 + gamma.unsqueeze(-1)) + beta.unsqueeze(-1)


class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad):
        return -grad


class CNNLSTM(nn.Module):
    def __init__(self, width=128, hidden=128, n_classes=5, use_demo=False, demo_dim=16, gate=False, film=False,
                 adv=False):
        super().__init__()
        self.stem = conv_stem(width)
        self.lstm = nn.LSTM(width, hidden, batch_first=True, bidirectional=True)
        self.pool = AttentionPool(2 * hidden)
        self.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(2 * hidden + (demo_dim if use_demo else 0), n_classes))
        self.demo = nn.Sequential(nn.Linear(2, demo_dim), nn.ReLU()) if use_demo else None
        self.gate = LeadGate() if gate else None
        self.film = FiLM(width) if film else None
        self.adv = nn.Sequential(nn.Linear(2 * hidden, 64), nn.ReLU(), nn.Linear(64, 1)) if adv else None
        self.adv_logit = None

    def forward(self, x, demo=None):
        if self.gate is not None:
            x = self.gate(x, demo)
        h = self.stem(x)
        if self.film is not None:
            h = self.film(h, demo)
        h, _ = self.lstm(h.transpose(1, 2))
        z = self.pool(h)
        if self.adv is not None:
            self.adv_logit = self.adv(GradReverse.apply(z)).squeeze(-1)
        if self.demo is not None:
            z = torch.cat([z, self.demo(demo)], dim=1)
        return self.head(z)


BASE = {'demo': False, 'weighted': False, 'gate': False, 'film': False, 'norm': 'global', 'adv': False}
VARIANTS = {
    'base': {}, 'demo': {'demo': True}, 'groupw': {'weighted': True}, 'demo_groupw': {'demo': True, 'weighted': True},
    'leadgate': {'gate': True}, 'film': {'film': True}, 'sexnorm': {'norm': 'sex'}, 'adv': {'adv': True},
}
VARIANTS = {k: {**BASE, **v} for k, v in VARIANTS.items()}
ADV_LAMBDA = 0.5

## Обучение (протокол недели 8)

In [ ]:
EPOCHS = 1 if SMOKE else 20
BATCH = 128
WD = 1e-2
LR = 1e-3
WIN = int(2.5 * FS)


def macro_auc(y, s):
    return float(np.mean([roc_auc_score(y[:, k], s[:, k]) for k in range(y.shape[1])]))


@torch.no_grad()
def predict(model, x, demo, batch=1024):
    model.eval()
    w = x.unfold(1, WIN, FS)
    n, n_win = w.shape[:2]
    flat = w.reshape(n * n_win, w.shape[2], WIN)
    d = demo.repeat_interleave(n_win, dim=0)
    out = torch.cat([torch.sigmoid(model(flat[s:s + batch], d[s:s + batch])) for s in range(0, len(flat), batch)])
    return out.reshape(n, n_win, -1).mean(dim=1).cpu().numpy()


def save_checkpoint(path, model, opt, sched, epoch, history, best, rng):
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'epoch': epoch, 'history': history, 'best': best, 'np_rng': rng.bit_generator.state,
                'torch_rng': torch.get_rng_state(),
                'cuda_rng': torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None}, path)


def restore_checkpoint(path, model, opt, sched, rng):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck['model'])
    opt.load_state_dict(ck['opt'])
    sched.load_state_dict(ck['sched'])
    rng.bit_generator.state = ck['np_rng']
    torch.set_rng_state(ck['torch_rng'])
    if ck['cuda_rng'] is not None:
        torch.cuda.set_rng_state_all(ck['cuda_rng'])
    return ck['epoch'], ck['history'], ck['best']


def train_epoch(model, opt, sched, rng, cfg, data):
    x_tr, y_tr, d_tr = data['train']
    model.train()
    order, losses = rng.permutation(len(x_tr)), []
    offsets = torch.arange(WIN, device=DEVICE)
    bce = nn.BCEWithLogitsLoss()
    bce_each = nn.BCEWithLogitsLoss(reduction='none')
    for s in range(0, len(order), BATCH):
        rows = torch.from_numpy(order[s:s + BATCH]).to(DEVICE)
        start = torch.from_numpy(rng.integers(0, x_tr.shape[1] - WIN + 1, len(rows))).to(DEVICE)
        xb = x_tr[rows[:, None], start[:, None] + offsets].permute(0, 2, 1)
        logits = model(xb, d_tr[rows])
        if cfg['weighted']:
            loss = (bce_each(logits, y_tr[rows]) * W_TRAIN[rows]).mean()
        else:
            loss = bce(logits, y_tr[rows])
        if cfg['adv']:
            loss = loss + ADV_LAMBDA * bce(model.adv_logit, d_tr[rows, 1])
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
        losses.append(loss.item())
    return float(np.mean(losses))


def run(variant, seed):
    rid = f'cnnlstm_{variant}_s{seed}' + ('_smoke' if SMOKE else '')
    if os.path.exists(RESULTS + rid + '.json'):
        return json.load(open(RESULTS + rid + '.json'))
    cfg = VARIANTS[variant]
    data = get_data(cfg['norm'])
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = CNNLSTM(use_demo=cfg['demo'], gate=cfg['gate'], film=cfg['film'], adv=cfg['adv']).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    steps = int(np.ceil(len(data['train'][0]) / BATCH))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=steps)
    start_epoch, history, best = 0, [], {'auc': -1.0, 'epoch': 0, 'state': None}
    if os.path.exists(CKPT + rid + '.pt'):
        start_epoch, history, best = restore_checkpoint(CKPT + rid + '.pt', model, opt, sched, rng)
        print(f'{rid}: продолжение с эпохи {start_epoch + 1}')
    x_va, y_va, d_va = data['val']
    for epoch in range(start_epoch, EPOCHS):
        t0 = time.perf_counter()
        loss = train_epoch(model, opt, sched, rng, cfg, data)
        val_auc = macro_auc(y_va.cpu().numpy(), predict(model, x_va, d_va))
        history.append({'epoch': epoch + 1, 'train_loss': loss, 'val_macro_auc': val_auc,
                        'sec': time.perf_counter() - t0})
        if val_auc > best['auc']:
            best = {'auc': val_auc, 'epoch': epoch + 1,
                    'state': {k: v.detach().clone() for k, v in model.state_dict().items()}}
        save_checkpoint(CKPT + rid + '.pt', model, opt, sched, epoch + 1, history, best, rng)
    model.load_state_dict(best['state'])
    s_val = predict(model, *data['val'][::2])
    s_test = predict(model, *data['test'][::2])
    np.savez(RESULTS + rid + '.npz', s_val=s_val, s_test=s_test, val_idx=idx['val'], test_idx=idx['test'])
    torch.save(best['state'], RESULTS + rid + '_weights.pt')
    info = {'run_id': rid, 'variant': variant, 'seed': seed, 'config': cfg, 'use_demo': cfg['demo'],
            'weighted': cfg['weighted'], 'best_epoch': best['epoch'], 'val_macro_auc': best['auc'],
            'n_params': sum(p.numel() for p in model.parameters()), 'history': history,
            'train_minutes': sum(h['sec'] for h in history) / 60, 'env': ENV, 'smoke': SMOKE,
            'age_mean': float(AGE_MEAN), 'age_std': float(AGE_STD)}
    json.dump(info, open(RESULTS + rid + '.json', 'w'), indent=1)
    os.remove(CKPT + rid + '.pt')
    print(f"{rid}: эпоха {best['epoch']}, val macro-AUC {best['auc']:.4f}, {info['train_minutes']:.1f} мин")
    return info

In [ ]:
SEEDS = (42,) if SMOKE else (42, 1, 2, 3, 4)
for variant in VARIANTS:
    for seed in SEEDS:
        run(variant, seed)

## Сводка

Скачайте папку `thesis_ptbxl/results_nirm4/` целиком в `notebooks/cache/results_nirm4/` (старые файлы
будут заменены теми же).

In [ ]:
rows = [json.load(open(RESULTS + f)) for f in sorted(os.listdir(RESULTS)) if f.endswith('.json')]
for v in VARIANTS:
    vals = [r['val_macro_auc'] for r in rows if r['variant'] == v]
    print(f'{v:<12} val macro-AUC по seed: ' + ' '.join(f'{a:.4f}' for a in vals)
          + (f'  (среднее {np.mean(vals):.4f})' if vals else ''))